# Anharmonic Phonons From NVT Molecular Dynamics

[![badge](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfc/janus-core/blob/main/docs/source/tutorials/python/anharmonic_phonons.ipynb)

This tutorial shows how to generate standard TDEP (https://tdep-developers.github.io/tdep/) input files through `post_process_kwargs` while running an NVT molecular dynamics simulation with `janus-core`, and then use them to compute anharmonic phonon properties with TDEP.

The workflow is:

1. Build a unit cell and supercell.
2. Run a small NVT molecular dynamics simulation with `janus-core`.
3. Automatically generate standard TDEP input files.
4. Use TDEP to extract force constants and compute anharmonic phonon properties.



In [2]:
from pathlib import Path

from ase.build import bulk
from ase.io import write

from janus_core.calculations.md import NVT
from janus_core.calculations.single_point import SinglePoint



## Build The Unit Cell And Supercell

We first construct a rocksalt NaCl unit cell and expand it to a `2 x 2 x 2` supercell.

The unit-cell structure is used to generate `infile.ucposcar`, while the supercell structure is used both for the molecular dynamics simulation and to generate `infile.ssposcar`.



In [3]:
unit_cell = bulk("NaCl", "rocksalt", a=5.63, cubic=True)
supercell = unit_cell * (2, 2, 2)

write("unit_cell.cif", unit_cell)
write("supercell.cif", supercell)

Path("unit_cell.cif"), Path("supercell.cif")


(PosixPath('unit_cell.cif'), PosixPath('supercell.cif'))

## Run NVT Molecular Dynamics And Generate TDEP Inputs

We now run a small NVT molecular dynamics simulation using the supercell structure.

TDEP input generation is enabled through `post_process_kwargs`. This automatically writes the following files into `tdep-inputs/` after the MD run finishes:

- `infile.ucposcar`
- `infile.ssposcar`
- `infile.positions`
- `infile.forces`
- `infile.stat`
- `infile.meta`


In [5]:
single_point = SinglePoint(
    struct="supercell.cif",
    arch="mace_mp",
    model="small",
)

nvt = NVT(
    struct=single_point.struct,
    temp=300.0,
    steps=200,
    timestep=1.0,
    traj_every=20,
    stats_every=20,
    file_prefix="janus_results/NaCl-nvt-T300",
    post_process_kwargs={
        "tdep_compute": True,
        "tdep_unit_cell_file": "unit_cell.cif",
        "tdep_supercell_file": "supercell.cif",
        "tdep_output_dir": "janus_results/tdep-inputs",
    },
)

nvt.run()


Using Materials Project MACE for MACECalculator with /Users/junwen.yin/.cache/mace/20231210mace128L0_energy_epoch249model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.
Using head Default out of ['Default']


/Users/junwen.yin/research/8-tdep/janus-core/.venv/lib/python3.12/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/Users/junwen.yin/research/8-tdep/janus-core/.venv/lib/python3.12/site-packages/ase/md/langevin.py:110: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)
/Users/junwen.yin/research/8-tdep/janus-core/.venv/lib/python3.12/site-packages/ase/io/extxyz.py:320: UserWarning: Skipping unhashable information real_time
  warni

## Check The Generated TDEP Input Files

After the molecular dynamics simulation completes, the TDEP input files should be available in the `tdep-inputs/` directory.


In [6]:
required_files = [
    "infile.ucposcar",
    "infile.ssposcar",
    "infile.positions",
    "infile.forces",
    "infile.stat",
    "infile.meta",
]

for name in required_files:
    path = Path("janus_results/tdep-inputs") / name
    print(name, path.exists(), path)


infile.ucposcar True janus_results/tdep-inputs/infile.ucposcar
infile.ssposcar True janus_results/tdep-inputs/infile.ssposcar
infile.positions True janus_results/tdep-inputs/infile.positions
infile.forces True janus_results/tdep-inputs/infile.forces
infile.stat True janus_results/tdep-inputs/infile.stat
infile.meta True janus_results/tdep-inputs/infile.meta


## Run TDEP

We now move into the `tdep-inputs/` directory and run the TDEP commands needed to extract force constants and compute phonon dispersion relations and density of states. We assume you have already install TDEP using conda following https://github.com/tdep-developers/tdep/blob/main/INSTALL.md

In this example we:

1. Extract second-order force constants.
2. Copy `outfile.forceconstant` to `infile.forceconstant`.
3. Compute phonon dispersion relations and phonon density of states.


In [8]:
%cd janus_results/tdep-inputs
!pwd
!ls


/Users/junwen.yin/research/8-tdep/janus-core/docs/source/tutorials/python/janus_results/tdep-inputs
/Users/junwen.yin/research/8-tdep/janus-core/docs/source/tutorials/python/janus_results/tdep-inputs
infile.forces    infile.positions infile.stat
infile.meta      infile.ssposcar  infile.ucposcar


In [9]:
!conda activate tdep
!extract_forceconstants -rc2 100
!cp outfile.forceconstant infile.forceconstant
!phonon_dispersion_relations --dos -p
!gnuplot outfile.dispersion_relations.gnuplot_pdf 


CondaError: Run 'conda init' before 'conda activate'

 READ STRUCTURE AND SETUP CUTOFFS
 ... reading unitcell
 ... min cutoff:       2.81781
 ... max cutoff:       5.62995
 --> rc2 cutoff:     100.00000
 --> rc3 cutoff:      -1.00000
 --> rc4 cutoff:      -1.00000
 
 DETERMINING SYMMETRY OF INTERACTION TENSORS
 ... constructed distance tables (0.00477s)
 ... built singlets
 ... built pairs
 ... built prototype tuplets (0.00018s)
 ... expanded symmetry operations (0.26086s)
  ... reducing pairs                     100.0% |========================================|  0.00022s
  ... mapping pairs                      100.0% |========================================|  0.00371s
 N singlet shells:           2
          shell:  1   4
          shell:  2   4
 N pair shells:           6
    shell:           1           4   0.0000000000000000     
    shell:           2           4   0.0000000000000000     
    shell:           3          48   2.8149999999999999     
    shell:           4       

## Inspect The TDEP Outputs

After the TDEP commands finish, you should see output files such as:

- `outfile.forceconstant`
- `outfile.dispersion_relations`
- `outfile.dispersion_relations.pdf`
- `outfile.phonon_dos`
- `outfile.phonon_dos.hdf5`

These files can then be used to inspect the anharmonic phonon dispersion and density of states.
